# vcs_versioning smoke tests

Exercises the primary public API of `vcs_versioning` — the VCS-based versioning library
extracted from setuptools-scm. Five examples: the public exports, Version parsing and
comparison, Configuration defaults, a ScmVersion built by hand, and rendering versions with
the guess-next-dev scheme.

In [ ]:
# Example 1: Public API surface check — every advertised export is importable.
from vcs_versioning import (
    DEFAULT_LOCAL_SCHEME,
    DEFAULT_VERSION_SCHEME,
    Configuration,
    NonNormalizedVersion,
    PyProjectData,
    ScmVersion,
    Version,
    build_configuration_from_pyproject,
    infer_version_string,
)

print(f"DEFAULT_VERSION_SCHEME = {DEFAULT_VERSION_SCHEME!r}")
print(f"DEFAULT_LOCAL_SCHEME = {DEFAULT_LOCAL_SCHEME!r}")
print(f"Configuration: {Configuration}")
print(f"Version: {Version}")
print(f"ScmVersion: {ScmVersion}")

assert isinstance(DEFAULT_VERSION_SCHEME, str)
assert isinstance(DEFAULT_LOCAL_SCHEME, str)
assert callable(infer_version_string)
assert callable(build_configuration_from_pyproject)

In [ ]:
# Example 2: Version class — PEP 440 parsing and comparison.
from vcs_versioning import Version, NonNormalizedVersion

v1 = Version("1.2.3")
v2 = Version("1.2.4")
v_dev = Version("1.2.3.dev1")

print(f"v1={v1}, v2={v2}, v_dev={v_dev}")
print(f"v1 < v2: {v1 < v2}")
print(f"v_dev < v1: {v_dev < v1}")
print(f"v1.major={v1.major}, v1.minor={v1.minor}, v1.micro={v1.micro}")

# NonNormalizedVersion preserves original string but parses semantics.
nnv = NonNormalizedVersion("1.2.3-rc1")
print(f"NonNormalizedVersion: {nnv}")

assert v1 < v2
assert v_dev < v1
assert v1.major == 1 and v1.minor == 2 and v1.micro == 3

In [ ]:
# Example 3: Configuration construction with defaults.
from vcs_versioning import Configuration, DEFAULT_VERSION_SCHEME, DEFAULT_LOCAL_SCHEME

config = Configuration(
    dist_name="my-test-package",
    version_scheme=DEFAULT_VERSION_SCHEME,
    local_scheme=DEFAULT_LOCAL_SCHEME,
)

print(f"dist_name = {config.dist_name!r}")
print(f"version_scheme = {config.version_scheme!r}")
print(f"local_scheme = {config.local_scheme!r}")
print(f"fallback_version = {config.fallback_version!r}")

assert config.dist_name == "my-test-package"
assert config.version_scheme == DEFAULT_VERSION_SCHEME
assert config.local_scheme == DEFAULT_LOCAL_SCHEME

In [ ]:
# Example 4: ScmVersion — represents a version derived from VCS metadata.
from vcs_versioning import Configuration, ScmVersion, Version

config = Configuration(dist_name="demo-pkg")

# Construct a ScmVersion mimicking what a VCS backend would produce:
# tag 1.0.0, 3 commits since the tag, on the main node 'abc123', clean tree.
scm = ScmVersion(
    tag=Version("1.0.0"),
    distance=3,
    node="gabc123",
    dirty=False,
    branch="main",
    config=config,
)

print(f"tag = {scm.tag}")
print(f"distance = {scm.distance}")
print(f"node = {scm.node}")
print(f"dirty = {scm.dirty}")
print(f"branch = {scm.branch}")
print(f"exact = {scm.exact}")  # True only when distance is 0/None and not dirty

assert scm.distance == 3
assert scm.branch == "main"
assert not scm.exact

In [ ]:
# Example 5: Format a ScmVersion via the configured version scheme.
from vcs_versioning import Configuration, ScmVersion, Version
from vcs_versioning.overrides import GlobalOverrides
from vcs_versioning._version_schemes import format_version

with GlobalOverrides.from_env("VCS_VERSIONING_TEST"):
    config = Configuration(
        dist_name="demo-pkg",
        version_scheme="guess-next-dev",
        local_scheme="no-local-version",
    )

    # Exact-tag case: distance=0 -> rendered version equals the tag.
    exact = ScmVersion(tag=Version("2.5.0"), distance=0, node=None, dirty=False, config=config)
    rendered_exact = format_version(exact)
    print(f"exact-tag render: {rendered_exact!r} (exact={exact.exact})")

    # Post-tag case: 5 commits past 2.5.0 — guess-next-dev bumps to next dev release.
    post = ScmVersion(tag=Version("2.5.0"), distance=5, node="gdeadbee", dirty=False, config=config)
    rendered_post = format_version(post)
    print(f"post-tag render: {rendered_post!r}")

assert rendered_exact == "2.5.0"
assert rendered_post != "2.5.0"
assert "dev" in rendered_post